# Version Control Patterns in Real Projects

This notebook explores how branching and commit structure reveal the workflow a
team is actually using. It combines the Version Control with Git concept
(branch topology, merge vs. fast-forward commits, commit shape) with
shell-automation-style Git queries executed through Python: build a small
synthetic repository that mimics a strategy, then inspect its history with the
same `git` calls a DevOps engineer uses to audit a real codebase.

Three strategies are modeled — feature-branch with an explicit merge commit,
trunk-based with a fast-forward, and squash-merge — so the structural
fingerprints of each become visible. It follows the foundational primer on
Version Control with Git.


## Key terminology

- **Feature branch** — a short-lived branch cut from `main` for one piece of
  work, later merged back. Example: `feature/auth` cut from `main`, merged with
  `--no-ff`.
- **Merge commit** — a commit with two parents that ties a branch back into its
  base. Example: `git merge --no-ff feature/auth` produces a visible branch
  point in `git log --graph`.
- **Fast-forward** — advancing the base pointer along a linear chain of commits
  with no merge commit. Example: merging `trunk` into `main` with `--ff-only`
  leaves a straight line.
- **Squash merge** — collapsing all commits on a branch into a single commit on
  the base, retaining no branch history. Example: a PR merged with GitHub's
  "Squash and merge" collapses N commits into one.
- **Linear history** — a commit graph with no merge commits; every commit has
  exactly one parent. Example: `git log --merges` returns nothing.
- **Commit cadence** — the rhythm of commits over time; frequent small commits
  suggest trunk-based flow, sparse large commits suggest feature-branch flow.


In [ ]:
# last_verified: 2026-09-28 · version-control-with-git n/a
import os
import subprocess
import tempfile
from collections import Counter

def git(repo, *args):
    return subprocess.run(
        ['git', '-C', repo, *args],
        capture_output=True, text=True, check=True
    ).stdout.strip()

def make_repo(path):
    git(path, 'init', '-q', '-b', 'main')
    git(path, 'config', 'user.email', 'dev@example.com')
    git(path, 'config', 'user.name', 'Dev')
    return path

def commit(repo, fname, content, msg):
    fpath = os.path.join(repo, fname)
    os.makedirs(os.path.dirname(fpath) or '.', exist_ok=True)
    with open(fpath, 'w') as f:
        f.write(content)
    git(repo, 'add', fname)
    git(repo, 'commit', '-q', '-m', msg)


## Step 1 — Build sample repositories that mimic three workflows

Each builder creates commits and merges the same way a real team would. The
`analyze` function then inspects the resulting history with the same `git`
queries a DevOps engineer uses to audit a repository: branch list, merge-commit
count, and commit cadence.


In [ ]:
def build_feature_branch_repo(repo):
    commit(repo, 'README.md', '# Project', 'initial commit')
    git(repo, 'checkout', '-q', '-b', 'feature/auth')
    commit(repo, 'auth.py', 'def login():\n    pass\n', 'add auth module')
    git(repo, 'checkout', '-q', 'main')
    git(repo, 'merge', '--no-ff', '--no-edit', 'feature/auth')
    commit(repo, 'README.md', '# Project\n', 'update readme after merge')

def build_squash_repo(repo):
    commit(repo, 'README.md', '# Project', 'initial commit')
    git(repo, 'checkout', '-q', '-b', 'feature/button')
    commit(repo, 'button.js', 'export function Button() { return null }', 'add button')
    commit(repo, 'button.js', 'export function Button() { return <button/> }', 'refine button')
    git(repo, 'checkout', '-q', 'main')
    git(repo, 'merge', '--squash', '--no-edit', 'feature/button')
    git(repo, 'commit', '-q', '-m', 'add button (squashed)')

def build_trunk_repo(repo):
    commit(repo, 'README.md', '# Project', 'initial commit')
    git(repo, 'checkout', '-q', '-b', 'hotfix')
    commit(repo, 'index.html', '<html></html>', 'add index')
    commit(repo, 'index.html', '<html>\n  <!-- header -->\n</html>', 'add header')
    git(repo, 'checkout', '-q', 'main')
    git(repo, 'merge', '--ff-only', 'hotfix')

def analyze(repo):
    branches = git(repo, 'branch', '--list').splitlines()
    merges = git(repo, 'log', '--merges', '--oneline').splitlines()
    total = int(git(repo, 'rev-list', '--count', 'HEAD'))
    log_dates = git(repo, 'log', '--reverse', '--pretty=format:%ad', '--date=short')
    per_day = Counter(log_dates.splitlines()) if log_dates else {}
    return {
        'branches': sorted(b.strip().lstrip('* ') for b in branches),
        'branch_count': len(branches),
        'merge_commits': len([m for m in merges if m]),
        'total_commits': total,
        'commits_per_day': dict(per_day),
        'first_commit': min(per_day) if per_day else None,
        'last_commit': max(per_day) if per_day else None,
    }

def detect_pattern(info):
    if info['merge_commits'] >= 1:
        return 'feature-branch (explicit merge commits)'
    if info['merge_commits'] == 0:
        return 'trunk-based / squash-merge (linear history)'
    return 'unclassified'


## Step 2 — Inspect Pattern A: feature-branch with a merge commit

A merge commit (a commit with two parents) is the clearest fingerprint of a
feature-branch workflow. We cut a branch, commit on it, then merge it back with
`--no-ff` so the branch topology survives in the graph.


In [ ]:
with tempfile.TemporaryDirectory() as repo:
    make_repo(repo)
    build_feature_branch_repo(repo)
    info = analyze(repo)

print("Branch list:    ", info['branches'])
print("Branch count:    ", info['branch_count'])
print("Merge commits:   ", info['merge_commits'])
print("Total commits:   ", info['total_commits'])
print("Commits per day: ", info['commits_per_day'])
print("First commit:    ", info['first_commit'])
print("Last commit:     ", info['last_commit'])
print("Detected pattern:", detect_pattern(info))


## Step 3 — Inspect Pattern B: trunk-based, linear history

A trunk-based repository leaves behind no merge commits. `git log --merges` is
empty and every commit has exactly one parent, so the history reads as a straight
line. The signal: a zero merge-commit count with commits landing directly on the
mainline.


In [ ]:
with tempfile.TemporaryDirectory() as repo:
    make_repo(repo)
    build_trunk_repo(repo)
    info = analyze(repo)

print("Branch list:    ", info['branches'])
print("Branch count:    ", info['branch_count'])
print("Merge commits:   ", info['merge_commits'])
print("Total commits:   ", info['total_commits'])
print("Commits per day: ", info['commits_per_day'])
print("Detected pattern:", detect_pattern(info))


## Step 4 — Inspect Pattern C: squash-merge flow

A squash-merge collapses a feature branch into a single commit on the base.
Like trunk-based, it leaves no merge commits — but it can be distinguished by the
absence of retained branch history and the shape of the resulting messages. The
fingerprint is the same as trunk-based (zero merges), which is exactly why
squash-merge teams usually pair this with branch-protection rules and PR checks
to compensate.


In [ ]:
with tempfile.TemporaryDirectory() as repo:
    make_repo(repo)
    build_squash_repo(repo)
    info = analyze(repo)

print("Branch list:    ", info['branches'])
print("Branch count:    ", info['branch_count'])
print("Merge commits:   ", info['merge_commits'])
print("Total commits:   ", info['total_commits'])
print("Detected pattern:", detect_pattern(info))


## Comparison

The table below summarizes the structural fingerprints that distinguish the
three workflows. Pointing `analyze()` at any real repository path reproduces
these numbers locally.

| Workflow | Merge commits | Linear history | Typical branch count | Cadence fingerprint |
|---|---|---|---|---|
| Feature-branch | 1+ | no | grows with open features | clustered around PR merge days |
| Trunk-based | 0 | yes | 1–2 | frequent, evenly spaced |
| Squash-merge | 0 | yes | transient (deleted after merge) | one commit per PR |


## Verify

This notebook confirms:

1. Feature-branch workflows leave merge commits — `git log --merges` is
   non-empty and multiple branches are visible.
2. Trunk-based and squash-merge workflows produce linear history with zero merge
   commits.
3. Commit cadence — the distribution of commits across days — is a secondary
   signal that complements merge-commit counts.

Because squash and trunk-based both read as "linear," a repository should be
inspected with both structural (merge count) and policy (branch protection, PR
checks) signals before a conclusion is trusted.

## What I'll cover next

Pointing these same `git` queries at a real repository path, adding
authorship and file-ownership analysis, and wiring the pattern detector into a
CI check that gates on merge-commit hygiene.
